# AG05 · Memory & Context Engineering

**Track:** Agents (lesson 5 of 15) · **Time:** ~2.5 h · **Mode:** runs OFFLINE for free, or LIVE with `ANTHROPIC_API_KEY`
**Prereqs:** AG02 (agent loop), AG03 (tools), AG04 (planning) · **You will build:** a long-running travel concierge that survives 50 turns inside a fixed token budget, using clearing, compaction and a long-term memory store

---
## 🎯 Why this matters
An agent's "memory" is whatever you put in the next prompt, nothing more. The transcript grows every turn, every turn re-sends it,
so **cost grows quadratically with conversation length** and quality often degrades before the window is even full
(models attend less reliably to material buried in a very long context; see Liu et al., *Lost in the Middle*).
*Context engineering* is the discipline of deciding, on every call, **what is the smallest set of tokens that makes the right next action likely**.

**Real scenario.** A family travel concierge assistant plans a ski trip over dozens of messages. Early on the customer says
"my son is allergic to shellfish". Forty turns and many tool results later, the agent recommends a seafood restaurant.
That is a *memory* bug, and it is a product-safety bug, not a cosmetic one.

## 🧭 Learning objectives
By the end you can:
1. Treat context as a budget: measure per-turn context size, cumulative input tokens and cost for a conversation.
2. Implement and compare four context strategies (full history, sliding window, tool-result clearing + compaction, external notes) on **recall of seeded facts**.
3. Build a long-term memory store with explicit **write and read policies** (keyed updates, validation, retrieval under a token cap).
4. Explain why prompt caching rewards stable prefixes, and *measure* prefix reuse for each strategy.
5. Diagnose three memory failures: orphaned tool results, stale or poisoned memories, and summary drift.

In [ ]:
# Setup: runs LIVE with ANTHROPIC_API_KEY, otherwise OFFLINE (scripted replay, free)
import sys, pathlib, subprocess
_find = lambda: next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "agentkit").is_dir()), None)
_root = _find()
if _root is None:                        # Colab / standalone download: fetch the repo helper package
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/nexageapps/AI.git", "_ai_repo"], check=False)
    _root = pathlib.Path("_ai_repo").resolve()
sys.path.insert(0, str(_root))
from agentkit import setup
setup()

## 1 · Concept: context is a scarce budget

```
  system prompt ─┐
  long-term notes ├─▶ [ context for THIS call ] ─▶ model ─▶ next action
  recent turns ───┤        (you choose what goes in)
  tool results ───┘
```

Four levers, from cheapest to most lossy:

| Lever | What it does | Loses information? |
|---|---|---|
| **Clear stale tool results** | Replace old, bulky tool outputs by a placeholder once the agent has acted on them | Little: the *decision* is still in the assistant turn |
| **Compaction** | Ask a model to summarise old turns into a few lines, keep the last few turns verbatim | Yes, whatever the summary omits |
| **External notes / memory** | The agent *writes* durable facts to a store outside the window and *reads* them back when needed | Only what it chose not to write |
| **Sub-context isolation** | A sub-agent does a messy job in its own window and returns a short result (AG11) | By design |

Two cost facts to keep in mind. (1) Input tokens are billed **on every call**, so a context of size *c* at each of *n* turns costs about *n·c*, not *c*.
(2) **Prompt caching** lets the provider reuse the computation for an identical *prefix* of your prompt at a lower price; the exact discount and
rules are on the pricing/docs pages, so check them. The design implication is independent of the numbers:
*anything that rewrites the beginning of the prompt invalidates the cache from that point on.* We will measure this below.

**The simulation.** To compare strategies fairly we need the same 50-turn conversation each time. We generate it with a fixed seed:
five *durable facts* are said by the customer at known turns, buried among routine turns that each carry a bulky tool result.
At the end we "probe" each strategy: *is each fact still visible in the context the model would receive?*
That is a sound test because a model cannot use information that is not in its prompt.

## 2 · Build it: the conversation, the strategies, the harness
First the measuring stick. Token counts here use the rough `characters / 4` heuristic for English text; it is an *estimate* (use the API's
token-counting facility for real budgets) but it is monotone and good enough to compare strategies.

In [ ]:
import json, math, re, random
from dataclasses import dataclass, replace
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from agentkit import tool, ToolBox, get_llm, say, call, run_agent, PolicyLLM
from agentkit.trace import PRICES
from agentkit.llm import DEFAULT_MODEL

def est_tokens(x) -> int:
    """Rough token estimate: ~4 characters per token for English prose/JSON."""
    s = x if isinstance(x, str) else json.dumps(x, default=str)
    return max(1, math.ceil(len(s) / 4))

BUDGET = 4000          # tokens we allow ourselves per call (deliberately small so the effects show in 50 turns)
SYSTEM = ("You are Kiri, a travel concierge for a New Zealand agency. Plan trips step by step, use tools for prices and availability, "
          "never guess, and respect every constraint the customer has given you, including allergies, budget and airports.")

@dataclass
class Turn:
    n: int
    user: str
    tool: str          # bulky tool result for this turn ("" if none)
    assistant: str

# (turn number, utterance) for the five durable facts; the budget is *updated* at turn 44.
FACT_TURNS = {
    2:  "Hi, I'm Priya Raman. We're planning a ski trip to Queenstown in July.",
    6:  "Our hard budget cap is NZD 6,500 for everything.",
    11: "My son Kai is allergic to shellfish, so please remember that for any restaurant picks.",
    17: "We must fly out of Wellington, not Auckland.",
    44: "Update: we can stretch the budget to NZD 7,200 (new cap).",
}
# probe = (what we ask, substring that must be visible in the context to answer correctly)
PROBES = [("Who is the customer?", "Priya Raman"), ("Where are they going?", "Queenstown"),
          ("What is the CURRENT budget cap?", "7,200"), ("Any dietary allergy?", "shellfish"),
          ("Which airport do they depart from?", "Wellington")]

CUE = re.compile(r"\b(I'm|my|our|must|allergic|budget|prefer|remember|update)\b", re.I)   # words that signal a durable fact

def make_conversation(n_turns: int = 50, seed: int = 7) -> list[Turn]:
    rng = random.Random(seed)
    topics = ["lift ticket prices", "family lodges near the Remarkables", "rental car options", "ski school availability",
              "rental gear packages", "restaurant availability", "airport shuttle times", "weather outlook"]
    turns = []
    for n in range(1, n_turns + 1):
        if n in FACT_TURNS:
            turns.append(Turn(n, FACT_TURNS[n], "", "Noted, thank you. I will keep that in mind as we plan."))
            continue
        topic = rng.choice(topics)
        user = f"Please check {topic} for day {rng.randint(1, 7)}."
        assert not CUE.search(user), "routine turns must not contain durable-fact cue words"
        rows = [{"vendor": rng.choice(["AlpineCo", "Southern Lakes Ltd", "Coronet Group", "FernHill"]),
                 "sku": f"SKU-{rng.randint(1000, 9999)}", "price_nzd": round(rng.uniform(40, 600), 2),
                 "available": rng.random() > 0.3, "notes": "Terms apply; subject to availability; prices include GST."}
                for _ in range(14)]
        tool_blob = json.dumps({"query": topic, "results": rows})
        turns.append(Turn(n, user, tool_blob, f"Here are the best options I found for {topic}: " + "; ".join(
            f"{r['vendor']} at ${r['price_nzd']}" for r in rows[:3]) + ". Want me to shortlist any?"))
    return turns

CONVO = make_conversation()
tok_per_turn = [est_tokens(t.user) + est_tokens(t.tool) + est_tokens(t.assistant) for t in CONVO]
print(f"{len(CONVO)} turns | avg {np.mean(tok_per_turn):.0f} tokens/turn | whole transcript ≈ {sum(tok_per_turn):,} tokens | budget per call = {BUDGET:,}")
assert sum(tok_per_turn) > 5 * BUDGET, "the conversation must overflow the budget, otherwise there is nothing to engineer"

### The strategies
Each strategy has the same interface: `observe(turn)` after every completed turn, and `context()` returning the list of strings that would
be sent on the *next* call (system prompt first). The harness then measures size, cumulative billed input and recall.

* **Full history**: append everything. The default in most tutorials.
* **Sliding window**: keep the newest whole turns that fit the budget.
* **Clear tool results**: keep every user and assistant message, but blank old tool outputs (keep the most recent few). Cheap and surprisingly strong.
* **Compaction**: clear tool results *and*, when the context passes 80 % of budget, ask a model to fold the oldest turns into a short running summary.
* **Notes**: the agent writes durable facts to an external store (`MemoryStore`, built next), and the context is `system + notes + last few turns`.

The compaction summariser and the note-taker are *model calls*. Offline, a small Python policy stands in for Claude (it keeps sentences with
cue words; the note-taker uses regexes). **Live, the same code calls the real model**, and you should expect it to be both better and different.
The harness, budgets and metrics are real in both modes.

In [ ]:
class MemoryStore:
    """Long-term memory: keyed facts + lexical retrieval. Write policy lives in .write(), read policy in .search()/.render()."""
    def __init__(self):
        self.facts: dict[str, dict] = {}
        self.clock = 0

    def write(self, key: str, value: str, source: str = "agent") -> str:
        if not re.fullmatch(r"[a-z][a-z0-9_]{1,40}", key):
            raise ValueError("key must be snake_case, e.g. 'dietary_allergy'")
        value = value.strip()
        if not value or len(value) > 200:
            raise ValueError("value must be 1-200 characters: store a fact, not a transcript")
        if re.search(r"\d{13,19}", re.sub(r"[ -]", "", value)):
            raise ValueError("looks like a card number; refusing to store sensitive data")
        self.clock += 1
        old = self.facts.get(key)
        self.facts[key] = {"value": value, "ts": self.clock, "source": source, "history": (old["history"] + [old["value"]]) if old else []}
        return f"saved {key}" + (f" (updated; was '{old['value']}')" if old and old["value"] != value else "")

    def search(self, query: str, k: int = 3, min_score: float = 0.05) -> list[tuple[str, str, float]]:
        from sklearn.feature_extraction.text import TfidfVectorizer
        if not self.facts:
            return []
        keys = list(self.facts)
        docs = [f"{k.replace('_', ' ')} {self.facts[k]['value']}" for k in keys]
        vec = TfidfVectorizer(stop_words="english").fit(docs + [query])
        sims = (vec.transform(docs) @ vec.transform([query]).T).toarray().ravel()
        order = np.argsort(-sims)[:k]
        return [(keys[i], self.facts[keys[i]]["value"], float(sims[i])) for i in order if sims[i] >= min_score]

    def render(self, max_tokens: int = 300) -> str:
        lines, used = [], 0
        for k in sorted(self.facts, key=lambda k: -self.facts[k]["ts"]):       # newest first when over budget
            line = f"- {k}: {self.facts[k]['value']}"
            if used + est_tokens(line) > max_tokens: break
            lines.append(line); used += est_tokens(line)
        return "MEMORY (durable customer facts):\n" + "\n".join(lines)

EXTRACT = [("traveller", r"I'm ([A-Z][a-z]+ [A-Z][a-z]+)"), ("destination", r"trip to (\w+)"), ("budget_nzd", r"NZD ([\d,]+)"),
           ("dietary_allergy", r"allergic to (\w+)"), ("home_airport", r"fly out of (\w+)")]

def note_taker(turn: Turn, store: MemoryStore):
    """Offline stand-in for 'the model decides to call memory_write'. Live, the model does this through the tool (see §3)."""
    for key, pat in EXTRACT:
        m = re.search(pat, turn.user)
        if m: store.write(key, m.group(1), source=f"turn {turn.n}")

def summarizer_policy(messages, system, tools):
    """Offline stand-in for the summariser model: keep customer sentences that contain durable-fact cue words."""
    prompt = messages[-1]["content"]
    cap = int(re.search(r"MAX_LINES=(\d+)", prompt).group(1))
    prev, _, new = prompt.partition("NEW TURNS:")
    lines = [l[2:] for l in prev.splitlines() if l.startswith("- ")]
    lines += [l[len("user: "):] for l in new.splitlines() if l.startswith("user: ") and CUE.search(l)]
    lines = list(dict.fromkeys(lines))[-cap:]
    return say("\n".join("- " + l for l in lines))

def flat(t: Turn, clear_tool: bool = False) -> list[str]:
    out = [f"user: {t.user}"]
    if t.tool: out.append("tool_result: [cleared: stale tool output]" if clear_tool else f"tool_result: {t.tool}")
    out.append(f"assistant: {t.assistant}")
    return out

class Strategy:
    name = "base"
    def __init__(self, budget: int = BUDGET):
        self.budget, self.turns, self.extra_tokens = budget, [], 0   # extra_tokens = billed tokens for side calls (summariser)
    def observe(self, t: Turn): self.turns.append(t)
    def context(self) -> list[str]: raise NotImplementedError
    def tokens(self) -> int: return sum(est_tokens(s) for s in self.context())

class FullHistory(Strategy):
    name = "full history"
    def context(self): return [SYSTEM] + [s for t in self.turns for s in flat(t)]

class SlidingWindow(Strategy):
    name = "sliding window"
    def context(self):
        keep, used = [], est_tokens(SYSTEM)
        for t in reversed(self.turns):                       # whole turns only (see 4a for why)
            cost = sum(est_tokens(s) for s in flat(t))
            if used + cost > self.budget: break
            keep.append(t); used += cost
        return [SYSTEM] + [s for t in reversed(keep) for s in flat(t)]

class ClearToolResults(Strategy):
    name = "clear tool results"
    def __init__(self, budget=BUDGET, keep_tools: int = 3):
        super().__init__(budget); self.keep_tools = keep_tools
    def context(self):
        n = len(self.turns)
        return [SYSTEM] + [s for i, t in enumerate(self.turns) for s in flat(t, clear_tool=(i < n - self.keep_tools))]

class Compaction(ClearToolResults):
    name = "clear + compaction"
    def __init__(self, llm, budget=BUDGET, trigger=0.8, keep_recent=6, cap_lines=12):
        super().__init__(budget); self.llm, self.trigger, self.keep_recent, self.cap = llm, trigger, keep_recent, cap_lines
        self.summary = ""
        self.compactions = 0
    def context(self):
        head = [SYSTEM] + ([f"SUMMARY OF EARLIER CONVERSATION:\n{self.summary}"] if self.summary else [])
        n = len(self.turns)
        return head + [s for i, t in enumerate(self.turns) for s in flat(t, clear_tool=(i < n - self.keep_tools))]
    def observe(self, t):
        super().observe(t)
        if self.tokens() > self.trigger * self.budget and len(self.turns) > self.keep_recent:
            old, self.turns = self.turns[:-self.keep_recent], self.turns[-self.keep_recent:]
            prompt = (f"Summarise the conversation so far for a travel concierge who will lose the original messages. "
                      f"Keep every customer constraint and preference. MAX_LINES={self.cap}\n"
                      f"PREVIOUS SUMMARY:\n{self.summary}\nNEW TURNS:\n" + "\n".join(s for o in old for s in flat(o, clear_tool=True)))
            reply = self.llm.chat([{"role": "user", "content": prompt}], system="ROLE:summarizer")
            self.summary = reply.text
            self.extra_tokens += est_tokens(prompt) + est_tokens(reply.text)
            self.compactions += 1

class Notes(Strategy):
    name = "notes + recent"
    def __init__(self, budget=BUDGET, keep_recent=6):
        super().__init__(budget); self.keep_recent = keep_recent; self.store = MemoryStore()
    def observe(self, t):
        super().observe(t); note_taker(t, self.store)
    def context(self):
        n = len(self.turns)
        recent = self.turns[-self.keep_recent:]
        return [SYSTEM, self.store.render()] + [s for i, t in enumerate(recent) for s in flat(t, clear_tool=(i < len(recent) - 2))]

print("strategies defined:", [c.name for c in (FullHistory, SlidingWindow, ClearToolResults, Compaction, Notes)])

### The harness
For every turn we record (a) context size for the next call, (b) the *billed* input if that call were made (we bill each turn's context once,
plus any summariser calls), and (c) prefix reuse: the fraction of this turn's context that is an identical leading prefix of the previous turn's.
A provider-side prompt cache can only help on that identical-prefix fraction. Finally we probe for recall.

In [ ]:
def common_prefix_tokens(a: list[str], b: list[str]) -> int:
    n = 0
    for x, y in zip(a, b):
        if x != y: break
        n += est_tokens(x)
    return n

def run_conversation(strategy: Strategy, turns: list[Turn]) -> dict:
    sizes, reuse, prev = [], [], []
    billed = 0
    for t in turns:
        strategy.observe(t)
        ctx = strategy.context()
        size = sum(est_tokens(s) for s in ctx)
        sizes.append(size); billed += size
        reuse.append(common_prefix_tokens(prev, ctx) / size); prev = ctx
    text = "\n".join(strategy.context())
    visible = {q: (exp in text) for q, exp in PROBES}
    return {"name": strategy.name, "sizes": sizes, "peak": max(sizes), "billed_input": billed + strategy.extra_tokens,
            "recall": sum(visible.values()) / len(PROBES), "visible": visible, "prefix_reuse": float(np.mean(reuse[1:])),
            "obj": strategy}

summ_llm = get_llm(policy=summarizer_policy)
results = [run_conversation(s, CONVO) for s in
           (FullHistory(), SlidingWindow(), ClearToolResults(), Compaction(summ_llm), Notes())]

price_in = PRICES[DEFAULT_MODEL][0]
print(f"{'strategy':<20}{'peak ctx':>9}{'billed input':>14}{'cost*':>9}{'recall':>8}{'prefix reuse':>14}")
for r in results:
    print(f"{r['name']:<20}{r['peak']:>9,}{r['billed_input']:>14,}{'$%.3f' % (r['billed_input'] * price_in / 1e6):>9}{r['recall']:>8.0%}{r['prefix_reuse']:>14.0%}")
print(f"* uncached input only, at the PRICES entry for {DEFAULT_MODEL}; illustrative.")

R = {r["name"]: r for r in results}
assert R["full history"]["peak"] > 5 * BUDGET                      # overflows
assert R["full history"]["recall"] == 1.0                           # ...but remembers everything
assert R["sliding window"]["peak"] <= BUDGET and R["sliding window"]["recall"] <= 0.4   # fits, forgets
assert R["clear + compaction"]["peak"] <= BUDGET * 1.05 and R["clear + compaction"]["recall"] >= 0.8
assert R["notes + recent"]["peak"] <= BUDGET and R["notes + recent"]["recall"] == 1.0
assert R["clear tool results"]["recall"] == 1.0 and R["clear tool results"]["peak"] < 0.25 * R["full history"]["peak"]
assert R["full history"]["prefix_reuse"] > R["sliding window"]["prefix_reuse"]   # caching rewards append-only prompts
assert R["clear + compaction"]["obj"].compactions >= 1

### 🔍 Read the numbers
Look at the table as an engineer would:
* **Full history** remembers everything and is the *most cache-friendly* (append-only), but its context is unbounded and its cumulative bill is huge.
* **Sliding window** is the classic mistake: it fits the budget, and it silently forgets the allergy. It also rewrites the front of the prompt on every turn, which defeats prefix caching.
* **Clearing tool results** alone removes most of the bulk (tool output dominated the transcript) and loses nothing here, since the facts were in user turns. But look at its curve: it still *climbs* with every turn and ends slightly over budget. It delays the problem; it does not solve it.
* **Compaction** and **notes** hold the budget *and* keep the facts. They differ in *where* the facts live: in a summary the model wrote (lossy, may drift), or in a store the agent wrote deliberately (inspectable, editable, shared across sessions).

Plot it; the shape of these curves is worth remembering.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for r in results:
    ax.plot(range(1, len(CONVO) + 1), r["sizes"], label=r["name"], lw=2)
ax.axhline(BUDGET, color="k", ls="--", lw=1); ax.text(1, BUDGET * 1.05, "budget", fontsize=8)
for n in FACT_TURNS: ax.axvline(n, color="grey", alpha=.25)
ax.set_yscale("log"); ax.set_xlabel("turn (grey lines = a durable fact was stated)"); ax.set_ylabel("context tokens (log)")
ax.set_title("Context size per turn by strategy"); ax.legend(fontsize=8, ncol=2); plt.tight_layout(); plt.show()

## 3 · Use it: a long-term memory the *agent* controls (across sessions)
In §2 the note-taker was Python. In a real agent, **memory is just two more tools** and the model decides when to call them.
Your job is the *policy* around them:

| Policy | Question it answers | Our choice |
|---|---|---|
| **Write** | What deserves to be remembered? | Durable, user-stated facts and preferences; one fact per key; never secrets; ≤200 chars |
| **Update** | What if the fact changes? | Same key overwrites; old value kept in `history` for audit |
| **Read** | When and how much to recall? | Pull-based `memory_search` (top-k above a similarity floor) + a small `render()` in the prompt |
| **Forget** | How does it expire or get corrected? | Keyed delete/overwrite; expose to the user (privacy) |

Session 1 captures the allergy. Session 2 is a *new* `run_agent` call with an empty transcript: only the store survives.
(Our retrieval is TF-IDF, i.e. lexical; production systems often add embeddings, and agents often combine key lookup with semantic search.
AG06 builds the retrieval machinery properly.)

In [ ]:
store = MemoryStore()

@tool
def memory_write(key: str, value: str) -> str:
    """Save one durable fact about the customer for future sessions. Use for allergies, constraints, preferences and budgets the customer
    states; do NOT save small talk, tool output, or secrets such as card numbers. Writing an existing key replaces its value.

    Args:
        key: snake_case name for the fact, e.g. "dietary_allergy" or "home_airport".
        value: The fact itself, under 200 characters.
    """
    return store.write(key, value)

@tool
def memory_search(query: str) -> str:
    """Search saved customer facts. Call this before recommending restaurants, flights or budgets-sensitive options.

    Args:
        query: What you want to recall, in plain words, e.g. "food allergies".
    """
    hits = store.search(query)
    return "\n".join(f"{k}: {v}" for k, v, _ in hits) or "No saved facts match."

mem_tools = ToolBox(memory_write, memory_search)

# SESSION 1 (recorded trace for offline mode)
llm = get_llm(script=[
    call("memory_write", _say="Saving that safety-critical detail.", key="dietary_allergy", value="Son Kai is allergic to shellfish"),
    say("Noted: Kai's shellfish allergy is saved, and I will apply it to every restaurant suggestion."),
])
s1 = run_agent(llm, mem_tools, "My son Kai is allergic to shellfish. Please remember that.", system=SYSTEM)
print("session 1 ->", s1.final_text)

# SESSION 2: next day, brand-new transcript
llm = get_llm(script=[
    call("memory_search", _say="Let me check what I know about dietary needs first.", query="food allergies dietary"),
    say("Because Kai is allergic to shellfish, I'd suggest burger bars or the grill room rather than the seafood restaurants."),
])
s2 = run_agent(llm, mem_tools, "Find us somewhere for dinner tonight in Queenstown.", system=SYSTEM)
s2.tracer.show()
assert store.facts["dietary_allergy"]["value"].endswith("shellfish")
assert s2.tracer.n_tool_calls("memory_search") == 1 and "shellfish" in s2.final_text.lower()
assert len(s2.messages) < 6        # session 2's transcript did not carry session 1 inside it: the store did

### Scratchpad files for long tasks (the same idea, different lifetime)
Memory above is *across sessions*. A **scratchpad** is memory *within one long task*: the agent keeps a `PLAN.md` or `progress.md`
(checklist, decisions, open questions) outside the transcript and re-reads it after compaction or a restart. It is the cheapest
way to make an agent resumable, and it is exactly what coding agents do. In the harness it is just one more entry in the store
(`store.write("task_plan", "1. flights ✔ 2. lodge ✔ 3. ski school (waiting on Priya)")`), or a file with a read/write tool.

## 4 · Break it
Memory bugs are quiet: nothing crashes, the agent is just subtly wrong later. Three that bite real systems.

### 4a · Truncation that orphans a `tool_result`
The API requires every `tool_result` to answer a `tool_use` in the *preceding* assistant message. If you trim history by *message count*
you can cut between the two. The provider rejects that transcript, or worse, the model sees a result with no question.
The fix: trim at **turn boundaries** (we did, above), and validate before sending.

In [ ]:
def validate_transcript(messages: list[dict]) -> list[str]:
    """Return a list of protocol problems (empty list = well formed)."""
    problems, open_ids = [], set()
    for i, m in enumerate(messages):
        blocks = m["content"] if isinstance(m["content"], list) else []
        if m["role"] == "assistant":
            if open_ids: problems.append(f"msg {i}: tool_use {sorted(open_ids)} never got a result")
            open_ids = {b["id"] for b in blocks if b.get("type") == "tool_use"}
        else:
            for b in blocks:
                if b.get("type") == "tool_result":
                    if b["tool_use_id"] not in open_ids: problems.append(f"msg {i}: orphan tool_result for {b['tool_use_id']}")
                    open_ids.discard(b["tool_use_id"])
    return problems

transcript = [
    {"role": "user", "content": "Lift tickets for day 3?"},
    {"role": "assistant", "content": [{"type": "text", "text": "Checking."}, {"type": "tool_use", "id": "toolu_1", "name": "search_lifts", "input": {"day": 3}}]},
    {"role": "user", "content": [{"type": "tool_result", "tool_use_id": "toolu_1", "content": "[...]"}]},
    {"role": "assistant", "content": [{"type": "text", "text": "Adult day pass is NZD 189."}]},
    {"role": "user", "content": "And day 4?"},
]
assert validate_transcript(transcript) == []
naive = transcript[-3:]                  # "keep the last 3 messages": starts with a dangling tool_result
print("naive cut :", validate_transcript(naive))
aligned = transcript[-2:]                # cut at a turn boundary instead
print("aligned   :", validate_transcript(aligned))
assert validate_transcript(naive) and validate_transcript(aligned) == []

### 4b · Stale, conflicting and poisoned memories
*Append-only* memory is the obvious design and the wrong one. After the customer raises the budget, an append-only log says both
"6,500" and "7,200", and the model must guess which is current. A *keyed* store resolves it at write time. Memory is also an
**injection surface**: anything the agent writes can later be read as trusted context, so validate writes (and treat tool-derived text as untrusted).

In [ ]:
append_only = []
keyed = MemoryStore()
for t in CONVO:
    m = re.search(r"NZD ([\d,]+)", t.user)
    if m:
        append_only.append(f"budget is NZD {m.group(1)}"); keyed.write("budget_nzd", m.group(1), source=f"turn {t.n}")
print("append-only memory says:", append_only)
print("keyed memory says      :", keyed.facts["budget_nzd"]["value"], "| history:", keyed.facts["budget_nzd"]["history"])
assert len(append_only) == 2 and keyed.facts["budget_nzd"]["value"] == "7,200" and keyed.facts["budget_nzd"]["history"] == ["6,500"]

for bad_key, bad_val in [("note", "x" * 500), ("Card Number", "visa"), ("payment", "card 4111 1111 1111 1111")]:
    try:
        keyed.write(bad_key, bad_val); print("!! accepted", bad_key)
    except ValueError as e:
        print("rejected:", bad_key, "->", e)
assert "payment" not in keyed.facts

### 4c · Summary drift: compaction is lossy, and lossy-on-lossy compounds
Each compaction re-summarises the *previous* summary. If the summary has a tiny line cap (or the summariser is sloppy), facts fall off the end
one compaction at a time. Measure it by sweeping the summary cap and watching recall:

In [ ]:
for cap in (12, 4, 2):
    r = run_conversation(Compaction(get_llm(policy=summarizer_policy), cap_lines=cap), CONVO)
    print(f"summary cap = {cap:>2} lines -> recall {r['recall']:.0%}  (compactions: {r['obj'].compactions}, peak {r['peak']:,})")
tight = run_conversation(Compaction(get_llm(policy=summarizer_policy), cap_lines=2), CONVO)
roomy = run_conversation(Compaction(get_llm(policy=summarizer_policy), cap_lines=12), CONVO)
assert tight["recall"] < roomy["recall"]

> **Principle:** *put safety-critical constraints where compaction cannot touch them*: in a keyed memory re-injected on every call
> (or pinned in the system prompt), not only in prose that a summariser may paraphrase away. And evaluate memory like any other
> feature: seed facts, run long, probe. That is exactly what this lab did.

## 5 · Exercises
**Practice (30 min)**
1. Add a sixth fact ("we prefer window seats") at turn 23 and a sixth probe. Confirm every strategy still passes or fails for the reasons you predicted.
2. Change `BUDGET` to 2,500 and to 8,000 and re-run the table. At which budgets does *clear tool results* alone suffice?

**Stretch (60 min)**
3. Write `Notes` with a **read policy**: instead of rendering all notes, call `store.search(last_user_message)` and inject only the top 3.
   Seed a *distractor* fact and show where lexical retrieval misses a relevant note (e.g. "seafood" vs "shellfish"). What would you add?
4. Implement a **write-confidence rule**: only persist a fact if the user phrase contains a cue word *and* the key is in an allow-list. Measure what it blocks.

**Challenge (2 h, needs a key)**
5. Run `Compaction` LIVE (set `ANTHROPIC_API_KEY`; the summariser becomes a real model call). Compare recall with the offline policy. Did the model keep the allergy? Did it invent anything? Add a verification step: after summarising, check that every cue-word sentence from the dropped turns is still *semantically* present.

In [ ]:
# Starter for exercise 3 - fill in and make the assertions pass
class NotesWithSearch(Notes):
    name = "notes (searched)"
    def context(self):
        # TODO: take the latest user message from self.turns[-1].user, call self.store.search(...), render only those hits
        raise NotImplementedError

# s = NotesWithSearch(); [s.observe(t) for t in CONVO]
# store_hit = s.store.search("shellfish allergy")
# assert store_hit and store_hit[0][0] == "dietary_allergy"
# assert s.store.search("seafood allergy") == [] or print("lexical retrieval misses synonyms; what would fix it?") is None

## ✅ Checkpoint
1. A 50-turn agent resends its history on each call. Why does total input cost grow roughly *quadratically* with turns, and which two levers flatten it?
2. Why does a sliding window defeat prompt caching, while an append-only transcript does not?
3. Name one thing compaction preserves well and one thing it loses. Where should a "never recommend shellfish" constraint live instead?
4. Your memory store contains `budget: 6,500` and `budget: 7,200` as two separate entries. What went wrong in the *write policy*, and what is the fix?

<details><summary>Answers</summary>

1. Each call bills the whole context, so cost ≈ sum over turns of a context that itself grows linearly: O(n²). Flatten by clearing/compacting old context (smaller *c*) and by caching a stable prefix (cheaper per token).
2. Caches match an identical prefix. Append-only keeps the entire previous prompt as the prefix; a window drops the oldest turns from the front, so after the system prompt nothing lines up.
3. It preserves the gist and decisions, and loses precise details and rarely-restated constraints (and can drift when re-summarised). Safety constraints belong in a keyed memory re-injected every call (or the system prompt), validated on write.
4. The store was append-only instead of keyed. Writes should address a fact by key so an update replaces the old value (keeping history for audit).
</details>

## 📌 Key takeaways
- Context is a **budget**: measure size per turn and cumulative billed input, not just whether it fits the window.
- Cheapest first: **clear stale tool output**, then **compact**, then move durable facts to **external memory**.
- Memory needs **policies** (write, update, read, forget), and writes are an attack and error surface: validate them.
- Trim on **turn boundaries**; keep stable content at the **front** of the prompt for caching.
- Evaluate memory with **seeded facts and probes**; failures are silent otherwise.

## 📚 Further reading
- Anthropic Engineering, *Effective context engineering for AI agents* (2025), anthropic.com/engineering
- Liu et al., *Lost in the Middle: How Language Models Use Long Contexts* (2023), arXiv:2307.03172
- Packer et al., *MemGPT: Towards LLMs as Operating Systems* (2023), arXiv:2310.08560
- Park et al., *Generative Agents: Interactive Simulacra of Human Behavior* (2023), arXiv:2304.03442 (memory stream, retrieval by recency, importance and relevance)
- Anthropic API docs: *Prompt caching* (check current limits and pricing there)

**Next → AG06 · Agentic RAG** (retrieval as a tool: decompose, grade, retry, cite)